In [2]:
# 导入库
import pandas as pd
from bs4 import BeautifulSoup

# 加载CSV文件到Pandas DataFrame
try:
    df = pd.read_csv('/home/gaojiayu/wb_project/data/postings.csv')
    print("数据加载成功！")
    print(f"数据集包含 {df.shape[0]} 行和 {df.shape[1]} 列。")
except FileNotFoundError:
    print("错误: 文件未找到。请确保文件名和路径正确。")
    
# 显示 DataFrame 的前5行
df.head()

数据加载成功！
数据集包含 123849 行和 31 列。


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
# 显示 DataFrame 的信息摘要
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [4]:
# 计算并显示每列的缺失值数量
df.isnull().sum()

job_id                             0
company_name                    1719
title                              0
description                        7
max_salary                     94056
pay_period                     87776
location                           0
company_id                      1717
views                           1689
med_salary                    117569
min_salary                     94056
formatted_work_type                0
applies                       100529
original_listed_time               0
remote_allowed                108603
job_posting_url                    0
application_url                36665
application_type                   0
expiry                             0
closed_time                   122776
formatted_experience_level     29409
skills_desc                   121410
listed_time                        0
posting_domain                 39968
sponsored                          0
work_type                          0
currency                       87776
c

In [6]:
# 职位标题（'title'）和职位描述（'description'）通常都包含关键信息。将它们合并成一个新列 full_text
title_col = 'title'
description_col = 'description'

# 合并前，先处理这两列可能存在的缺失值，填充为空字符串
df[title_col] = df[title_col].fillna('')
df[description_col] = df[description_col].fillna('')

# 将标题和描述合并成一个新列 'full_text'，用空格隔开
df['full_text'] = df[title_col] + ' : ' + df[description_col]

print("已成功创建 'full_text' 列。")
df[['title', 'description', 'full_text']].head()

已成功创建 'full_text' 列。


,title,description,full_text
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Marketing Coordinator : Job descriptionA leadi...
1,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",Mental Health Therapist/Counselor : At Aspen T...
2,Assitant Restaurant Manager,The National Exemplar is accepting application...,Assitant Restaurant Manager : The National Exe...
3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,Senior Elder Law / Trusts and Estates Associat...
4,Service Technician,Looking for HVAC service tech with experience ...,Service Technician : Looking for HVAC service...


In [7]:
# 检查 'full_text' 列的缺失值
# 理论上，经过上一步处理后，这里应该没有缺失值
missing_full_text = df['full_text'].isnull().sum()
print(f"'full_text' 列中的缺失值数量: {missing_full_text}")

'full_text' 列中的缺失值数量: 0


In [8]:
# 定义一个函数来清洗HTML标签
def clean_html(text):
    # 确保输入是字符串类型
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

# 创建一个新列 'cleaned_text' 来存储清洗后的文本
df['cleaned_text'] = df['full_text'].apply(clean_html)

print("HTML 清洗完成，结果已存入 'cleaned_text' 列。")

# 查看清洗前后的对比
print("\n--- 清洗前后对比 ---")
print("清洗前 (full_text):")
print(df['full_text'].iloc[10]) # 随意选择一行查看
print("\n清洗后 (cleaned_text):")
print(df['cleaned_text'].iloc[10])

HTML 清洗完成，结果已存入 'cleaned_text' 列。

--- 清洗前后对比 ---
清洗前 (full_text):
Inside Customer Service Associate : Glastender Inc. is a family-owned manufacturer of commercial bar and restaurant equipment, known for its high-quality products and innovative solutions. With a strong commitment to the customer experience, Glastender has been serving the industry for over 50 years, providing establishments with state-of-the-art equipment and exceptional service.We are currently looking for an Inside Customer Service Associate who can communicate with outside customers by providing exceptional customer service by addressing their concerns and resolving issues promptly (inquiries, orders, and product information via phone and email). Qualified candidates would be able to perform and possess the following skills:Design bar equipment layouts using the best application of Glastender products.Compile and submit quotations, perform order verification, order entry, and complete detailed shop drawings for use 